# User Profiles & Memory

> **Time:** ~12 minutes | **Level:** Beginner

**Profiles** are facts that Reflexio automatically extracts from user interactions.
Every time you publish a conversation, profile extractors analyze it and store structured knowledge about the user.

Profile lifecycle: **Pending** → **Current** → **Archived**

- **Pending** — freshly extracted, awaiting confirmation or auto-promotion
- **Current** — active, returned by default in searches
- **Archived** — superseded or manually removed, kept for audit

We will use a fictional **Acme Electronics** support scenario with two customers: Alice and Bob.

In [ ]:
import uuid

from _display_helpers import (
    load_env,
    show_profiles,
    show_response,
    show_success,
)

from reflexio import InteractionData, ReflexioClient, Status

url, api_key = load_env()
client = ReflexioClient(url_endpoint=url, api_key=api_key)

# Unique run ID so multiple notebook runs don't collide
RUN_ID = uuid.uuid4().hex[:8]
ALICE = f"alice_{RUN_ID}"
BOB = f"bob_{RUN_ID}"
print(f"Run ID: {RUN_ID}")
print(f"Alice: {ALICE}")
print(f"Bob:   {BOB}")

## Seed: Publish Conversations for Two Users

We publish realistic multi-turn conversations so the profile extractors have material to work with.
Using `wait_for_response=True` ensures profiles are generated before we query them.

In [ ]:
# Alice: graphic designer shopping for a new computer
client.publish_interaction(
    user_id=ALICE,
    interactions=[
        InteractionData(
            role="User",
            content="Hi! I'm a graphic designer looking for a new laptop. I mainly use Adobe Creative Suite and Figma.",
        ),
        InteractionData(
            role="Agent",
            content="Great! For creative work I'd recommend either a MacBook Pro or a high-end Windows workstation. Do you have a platform preference?",
        ),
        InteractionData(
            role="User",
            content="Definitely Mac. I've been using macOS for years and all my workflow is built around it. My budget is around $2,000.",
        ),
        InteractionData(
            role="Agent",
            content="The MacBook Pro 14-inch with M3 Pro chip fits that budget perfectly. It handles Creative Suite and Figma with ease. Would you also like a keyboard recommendation?",
        ),
        InteractionData(
            role="User",
            content="Yes please! I need something quiet — I work in a shared studio and a loud mechanical keyboard would bother everyone.",
        ),
        InteractionData(
            role="Agent",
            content="I'd suggest the Logitech MX Keys. It's a slim, quiet keyboard with Mac-specific key layout and great for long design sessions.",
        ),
    ],
    source="web_chat",
    agent_version="v1.0",
    wait_for_response=True,
)
show_success("Published Alice's conversation")

In [ ]:
# Bob: software engineer looking for a development workstation
client.publish_interaction(
    user_id=BOB,
    interactions=[
        InteractionData(
            role="User",
            content="I'm a software engineer and I need a machine for compiling large C++ projects and running Docker containers.",
        ),
        InteractionData(
            role="Agent",
            content="For heavy compilation and containerized workloads, you'll want a machine with a fast multi-core CPU and at least 32 GB of RAM. What OS do you prefer?",
        ),
        InteractionData(
            role="User",
            content="Linux all the way. I run Arch on everything. I need the fastest CPU I can get — compile times are killing my productivity.",
        ),
        InteractionData(
            role="Agent",
            content="The AMD Ryzen 9 7950X or Intel Core i9-14900K would both be excellent. Would you like a full workstation build or just a CPU recommendation?",
        ),
        InteractionData(
            role="User",
            content="A full build would be helpful. I don't care about how it looks — just pure performance. No RGB, no glass panels.",
        ),
        InteractionData(
            role="Agent",
            content="Understood. I'll put together a no-frills workstation build focused on CPU throughput and memory bandwidth. I'll skip the aesthetics and focus on airflow and reliability.",
        ),
    ],
    source="web_chat",
    agent_version="v1.0",
    wait_for_response=True,
)
show_success("Published Bob's conversation")

## Get All Profiles for a User

Retrieve every profile extracted for a specific user, regardless of search query.

In [ ]:
resp = client.get_profiles(force_refresh=True, user_id=ALICE)
show_profiles(resp.user_profiles, title=f"All Profiles for Alice ({ALICE})")

In [ ]:
resp = client.get_profiles(force_refresh=True, user_id=BOB)
show_profiles(resp.user_profiles, title=f"All Profiles for Bob ({BOB})")

## Keyword Search (FTS)

> **Full-text search** matches exact keywords in profile content. Fast and precise — use when you know the specific term.

In [ ]:
from reflexio.models.config_schema import SearchMode

resp = client.search_profiles(
    user_id=ALICE,
    query="keyboard",
    top_k=5,
    search_mode=SearchMode.FTS,
)
show_profiles(resp.user_profiles, title="Keyword search (FTS): 'keyboard'")

## Semantic Search

> Reflexio matches **meaning**, not just keywords. You can search using natural language even if the exact words weren't used in the conversation.

In [ ]:
# Alice mentioned she's a "graphic designer" — search for a related concept
resp = client.search_profiles(
    user_id=ALICE,
    query="what kind of creative work does this user do?",
    top_k=5,
    threshold=0.3,
)
show_profiles(resp.user_profiles, title="Semantic search: 'creative work'")

## Filter by Status

Profiles go through a lifecycle: **Pending** → **Current** → **Archived**.

- `Status.CURRENT` — active profiles (default for most queries)
- `Status.PENDING` — newly extracted, not yet confirmed
- `Status.ARCHIVED` — superseded by newer data

Use `status_filter` to control which profiles are returned.

In [ ]:
from rich import print as rprint

# Default get_profiles returns current profiles (status=None means current)
resp = client.get_profiles(force_refresh=True, user_id=ALICE)
show_profiles(resp.user_profiles, title="Alice's Profiles (current by default)")

rprint(f"\nTotal profiles: [bold]{len(resp.user_profiles)}[/bold]")
rprint("[dim]Profiles start as current (status=None). After a rerun, old ones become ARCHIVED and new ones are PENDING.[/dim]")

## Profile Change Log

> Track when profiles are **created**, **updated**, or **removed** over time.

In [ ]:
resp = client.get_profile_change_log()
show_response(resp, title="Profile Change Log")

## (Optional) Manual Profile Generation

> Re-extract profiles from existing interactions after changing your config.

`rerun_profile_generation` re-processes **all** interactions for a user and outputs profiles
with **Pending** status so you can review them. This is useful after updating your
`ProfileExtractorConfig` to see how the new prompt performs on historical data.

In [ ]:
resp = client.rerun_profile_generation(
    user_id=ALICE,
    wait_for_response=True,
)
show_response(resp, title="Profile Re-generation Result")

In [ ]:
# Check Alice's profiles after re-generation — new PENDING entries should appear
resp = client.get_profiles(force_refresh=True,
    user_id=ALICE,
    status_filter=[Status.CURRENT, Status.PENDING],
)
show_profiles(resp.user_profiles, title="Alice's Profiles After Re-generation")

In [ ]:
# --- Cleanup: remove all data created by this notebook ---
client.delete_all_interactions()
client.delete_all_profiles()
show_success("Cleanup complete.")

## Summary & Next Steps

In this notebook you learned how to:

- **Seed** profile data by publishing multi-turn conversations
- **Retrieve** all profiles for a user with `get_profiles()`
- **Search** profiles semantically within a user or across all users
- **Filter** profiles by lifecycle status (Pending, Current, Archived)
- **Inspect** the profile change log for audit and debugging
- **Re-generate** profiles after config changes with `rerun_profile_generation()`

Next, explore the **Reflexio Cookbook** (`reflexio_cookbook.ipynb`) for advanced topics like playbook extraction, aggregation, and unified search.